In [1]:
%pip install boto3
%pip install scikit-learn

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import boto3

# Initialize the S3 client 
s3 = boto3.client('s3')

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Preresequite: Create `.env` file in this dir, then add:
- `AWS_ACCESS_KEY_ID="Your Key"`, 
- `AWS_SECRET_ACCESS_KEY="Your Secrete Key"`, 
- `AWS_DEFAULT_REGION="us-east-2"`

Note: This file is in the .gitignore so will need to look at AWS IAM for keys or contact maintainers of repo for access.

In [ ]:
import boto3
import pandas as pd
import io
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

# 1. Initialize the S3 client
s3 = boto3.client('s3')
bucket_name = 'cleaned-filtered-data'
file_key = 'V-Dem/V_Dem_Filtered.csv' 

# 2. Fetch the object
obj = s3.get_object(Bucket=bucket_name, Key=file_key)
df = pd.read_csv(io.BytesIO(obj['Body'].read()))

print(f"Loaded: {file_key}")
print(f"Columns available: {df.columns.tolist()}")

Loaded: V-Dem/V_Dem_Filtered.csv
Columns available: ['year', 'country_name', 'country_id', 'country_text_id', 'v2x_accountability', 'v2x_civlib', 'v2x_corr', 'v2x_electoral_integrity', 'v2x_polyarchy', 'v2x_regime', 'e_v2xel_locelec_5C', 'e_wbgi_gee', 'e_uds', 'e_gdp', 'e_gdppc', 'e_miinflat']


In [ ]:
# Sort to ensure the time-shift happens correctly within each country
df = df.sort_values(['country_id', 'year'])

# Target 1: Predict GDP Per Capita 3 years in the future (t+3)
df['target_gdp_3yr'] = df.groupby('country_id')['e_gdppc'].shift(-3)

# Target 2: Predict Backsliding (10% drop in Unified Democracy Score over 3 years)
df['target_backslide_3yr'] = (
    df.groupby('country_id')['e_uds'].shift(-3) < (df['e_uds'] * 0.9)
).astype(int)

# Drop rows at the end of the timeline that don't have future data yet
df_model = df.dropna(subset=['target_gdp_3yr', 'target_backslide_3yr'])

print(f"Dataset ready for training. Total records: {len(df_model)}")

Dataset ready for training. Total records: 21910


In [ ]:

# 1. Drop all 'Direct' indicators to find the 'Indirect' Red Flags
X_red_flags = df_model.drop(columns=[
    'year', 'country_name', 'country_id', 'country_text_id', 
    'target_gdp_3yr', 'target_backslide_3yr', 
    'e_gdppc', 'e_gdp', 'e_miinflat',         # Economic Leakage
    'e_uds', 'v2x_polyarchy', 'v2x_regime'   # Democracy Target Leakage
])

# 2. Train
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_red_flags, y, test_size=0.2, random_state=42
)
model_rf = RandomForestRegressor(n_estimators=100)
model_rf.fit(X_train_rf, y_train_rf)

# 3. Check Importance of Red Flags
importances_final = pd.Series(model_rf.feature_importances_, index=X_red_flags.columns)
print("True Predictive Red Flags:")
print(importances_final.sort_values(ascending=False))

True Predictive Red Flags:
v2x_accountability         0.395538
e_wbgi_gee                 0.200465
v2x_electoral_integrity    0.133270
v2x_civlib                 0.131679
v2x_corr                   0.106914
e_v2xel_locelec_5C         0.032133
dtype: float64


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Define Features (X) and the Backsliding Target (y)
# We drop all leaky democracy scores so the model finds the red flags
X_backslide = df_model.drop(columns=[
    'year', 'country_name', 'country_id', 'country_text_id', 
    'target_gdp_3yr', 'target_backslide_3yr', 
    'e_gdppc', 'e_gdp', 'e_miinflat',
    'e_uds', 'v2x_polyarchy', 'v2x_regime' 
])

y_backslide = df_model['target_backslide_3yr']

# 2. Split for Classification
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_backslide, y_backslide, test_size=0.2, random_state=42, stratify=y_backslide
)

# 3. Train the Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_c, y_train_c)

# 4. Check the Results
print("Backsliding Prediction Report:")
print(classification_report(y_test_c, clf.predict(X_test_c)))

Backsliding Prediction Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4382

    accuracy                           1.00      4382
   macro avg       1.00      1.00      1.00      4382
weighted avg       1.00      1.00      1.00      4382

